# Diversity Metrics, By Hand

Notebook 01 used the word "diversity" as a concept. This notebook turns it
into numbers — the same way you hand-built the Benjamini-Hochberg correction
in notebook 04 instead of importing it from a library. You'll compute the two
standard families of diversity metric yourself, with nothing but `numpy`,
on the real USA vs. Malawi dataset.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("real_gut_usa_malawi.csv")
genera = [c for c in df.columns if c not in ("sample_id", "country")]
print(df.shape, "->", df.country.value_counts().to_dict())
print(len(genera), "genera:", genera)

## 2. Alpha diversity: how rich is ONE sample?

**Alpha diversity** answers: within a single person's sample, how many
different genera are present, and how evenly is abundance spread across
them? A sample dominated 99% by one genus is "less diverse" than one split
evenly across ten, even if both technically contain the same ten genera.

### Shannon index

The Shannon index (from information theory — it's literally entropy) is:

$$H = -\sum_i p_i \ln(p_i)$$

where $p_i$ is genus $i$'s proportion of the sample (its relative abundance
as a fraction, not a percentage). Higher $H$ = more diverse. A sample with
one genus at 100% scores exactly 0.

In [ ]:
def shannon(row_pcts):
    p = np.asarray(row_pcts, dtype=float) / 100.0   # percent -> fraction
    p = p[p > 0]                                     # log(0) is undefined; 0-abundance genera contribute nothing anyway
    return -np.sum(p * np.log(p))

df["shannon"] = df[genera].apply(shannon, axis=1)
df.groupby("country")["shannon"].describe()[["mean", "std", "min", "max"]].round(3)

### YOUR TURN #1
The line `p = p[p > 0]` drops genera with zero abundance in that sample
before taking the log. **Predict, then check:** what would `np.log(0)`
actually return in Python, and why would including it break the sum?

In [ ]:
# Try it:
np.log(0)

### Simpson index

The Simpson index asks a slightly different question: *if you picked two
individual bacteria at random from this sample, what's the probability
they're the SAME genus?* Lower = more diverse (more genera to "collide"
against). It's common to report `1 - Simpson` so that, like Shannon, higher
numbers mean more diverse:

$$\text{Simpson} = \sum_i p_i^2 \qquad \text{diversity} = 1 - \text{Simpson}$$

In [ ]:
def simpson_diversity(row_pcts):
    p = np.asarray(row_pcts, dtype=float) / 100.0
    return 1 - np.sum(p ** 2)

df["simpson"] = df[genera].apply(simpson_diversity, axis=1)
df.groupby("country")["simpson"].describe()[["mean", "std", "min", "max"]].round(3)

## 3. Is the alpha-diversity difference real?

Same tool you already know from notebook 03 — a t-test on the two country
groups' Shannon scores.

In [ ]:
usa = df[df.country == "USA"]["shannon"]
malawi = df[df.country == "Malawi"]["shannon"]
t, p = stats.ttest_ind(usa, malawi)
print(f"USA Shannon mean={usa.mean():.3f}  Malawi Shannon mean={malawi.mean():.3f}  p={p:.2e}")

### EXPLAIN #1
*Which country has higher alpha diversity by this measure? Does that
surprise you given what you learned in notebook 01 about diet and fiber?*

> your answer here

## 4. Beta diversity: how different are TWO samples from each other?

**Beta diversity** shifts the question from "how rich is this one sample"
to "how different is sample A's community from sample B's community?" The
classic metric is **Bray-Curtis dissimilarity**:

$$BC_{A,B} = \frac{\sum_i |p_{A,i} - p_{B,i}|}{\sum_i (p_{A,i} + p_{B,i})}$$

It ranges from 0 (identical composition) to 1 (no genera shared at all).

In [ ]:
def bray_curtis(row_a, row_b):
    a = np.asarray(row_a, dtype=float)
    b = np.asarray(row_b, dtype=float)
    return np.sum(np.abs(a - b)) / np.sum(a + b)

# sanity check: a sample compared to itself should be exactly 0
sample0 = df.loc[0, genera].values
print("self-distance:", bray_curtis(sample0, sample0))

## 5. Within-country vs. between-country distance

If diet really does split these communities the way notebook 01 claims,
two USA samples should look more like each other than a USA sample looks
compared to a Malawi sample. Let's check, using a small random subsample
(all-pairs on 150 samples would be ~11,000 comparisons — plenty for a real
analysis, overkill for a teaching notebook).

In [ ]:
rng = np.random.default_rng(0)
usa_idx = df.index[df.country == "USA"].to_numpy()
mw_idx = df.index[df.country == "Malawi"].to_numpy()

def sample_pairs(idx_pool_a, idx_pool_b, n=200):
    dists = []
    for _ in range(n):
        i = rng.choice(idx_pool_a)
        j = rng.choice(idx_pool_b)
        if i == j:
            continue
        dists.append(bray_curtis(df.loc[i, genera].values, df.loc[j, genera].values))
    return np.array(dists)

within_usa = sample_pairs(usa_idx, usa_idx)
between = sample_pairs(usa_idx, mw_idx)

print(f"Within-USA Bray-Curtis:      mean={within_usa.mean():.3f}")
print(f"USA-vs-Malawi Bray-Curtis:   mean={between.mean():.3f}")

### EXPLAIN #2
*Compare the two mean distances above. Does the pattern match what you'd
predict from the Prevotella/Bacteroides diet story in notebook 01? What
would it mean, biologically, if within-USA distance had come out HIGHER
than USA-vs-Malawi distance?*

> your answer here

### 🔧 YOUR TURN #2
Compute `within_malawi` the same way (`sample_pairs(mw_idx, mw_idx)`) and
compare its mean to `within_usa`. Malawi only has 21 samples in this
dataset — does that change how much you trust its mean?

In [ ]:
# Your code here

## Done — diversity isn't magic, it's arithmetic

You've now hand-built both families of diversity metric that real
microbiome pipelines compute automatically. That's exactly the bridge to
the next notebook: `qiime diversity core-metrics-phylogenetic` — one QIIME2
command — outputs Shannon, Simpson, and Bray-Curtis (plus a few
phylogeny-aware variants) for an entire study in one step. Now you know
what's actually happening inside that command.

**Next:** `07_bioinformatics_tools_walkthrough.ipynb`.